# Using MCP Tools with LangChain

In this notebook, we will explore how create a reasoning action agent using tools exposed by a MCP server with LangChain.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

## Create Model Object

Use `ChatOllama` to connect to the Ollama local endpoint. 

In [ ]:
model = ChatOllama(model='llama3.1:8b')

## Launch MCP Server

Using `StdioServerParameters`, we can launch a MCP server in the background that we can communicating using `stdio`. You can check the Python file [math_server.py](math_server.py), this is a very simple Python file that uses `FastMCP` to expose functions as tools.

In [ ]:
server_params = StdioServerParameters(command="python", args=["math_server.py"])

## Create stdio Session

Once the MCP server is running, we can start the `stdio_client` session. The cell bellow queries the available MCP tools and displays them.

In [ ]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:

        await session.initialize()

        # Convert MCP tools to LangChain tools
        tools = await load_mcp_tools(session)
        print(f"\nAvailable tools: {tools}\n\n")

## Create Async ReAct Agent

We can now create a reasoning and act agent, to do this we are using the `create_react_agent` available as part of LangGraph. This function takes the model as well as the available tools as parameters. Then we call the agent, that first will reason and if needed will use the available tools.

In [ ]:
async def run_react_agent(query: str):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            agent = create_react_agent(model, tools)
            agent_response = await agent.ainvoke({"messages": query})
            return agent_response

Now, we can query the react agent. Note that in the response from the agent the tool square_root is called and the result is used as part of the answer.

In [ ]:
query = "what is the square root of 55?"
agent_response = await run_react_agent(query)
for m in agent_response['messages']:
    m.pretty_print()

Similarly, if we ask for a multiplication operation, the multiplication tool is called.

In [ ]:
query = "what is the result of 3.14 * 5?"
agent_response = await run_react_agent(query)
for m in agent_response['messages']:
    m.pretty_print()

Again, the division tools is called.

In [ ]:
query = "what is the result of 99 / 2.3?"
agent_response = await run_react_agent(query)
for m in agent_response['messages']:
    m.pretty_print()

Note that this agent refuses to answer when we query about a topic that the agent does not find a suitable tool.

In [ ]:
query = "what is the capital of Ireland?"
agent_response = await run_react_agent(query)
for m in agent_response['messages']:
    m.pretty_print()

----------
Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved.

SPDX-License-Identifier: MIT